In [3]:
# Import necessary libraries
import pandas as pd  # For data manipulation
import numpy as np  # For numerical operations
import matplotlib.pyplot as plt  # For creating visualizations
import seaborn as sns  # For enhanced visualizations
from sklearn.model_selection import train_test_split  # For splitting data into training and testing sets
from sklearn.linear_model import LinearRegression  # For linear regression modeling
from sklearn.metrics import mean_squared_error, r2_score  # For model evaluation metrics
import statsmodels.api as sm  # For detailed statistical analysis
from statsmodels.stats.outliers_influence import variance_inflation_factor  # For multicollinearity detection
from scipy import stats  # For statistical tests
import os  # For file operations

# Set the style for plots to make them more visually appealing
plt.style.use('seaborn-v0_8-darkgrid')  # Using seaborn's darkgrid style
sns.set_palette("Set2")  # Setting a color palette for visualizations

def load_data(file_path):
    """
    Load the CSV data file from the specified path
    
    Parameters:
    -----------
    file_path : str
        Path to the CSV file
        
    Returns:
    --------
    pandas.DataFrame or None
        Loaded data as a DataFrame if successful, None otherwise
    """
    print(f"Attempting to load data from: {file_path}")
    try:
        # Check for browser environment with window.fs API (for web-based environments)
        if 'window' in globals() and hasattr(window, 'fs'):
            # Read file content using window.fs API
            file_content = window.fs.readFile(file_path, {'encoding': 'utf8'})
            # Convert content to DataFrame using StringIO
            return pd.read_csv(pd.StringIO(file_content))
        # For regular Python environment (Jupyter notebook, Python script)
        else:
            # Directly read CSV file into DataFrame
            return pd.read_csv(file_path)
    except Exception as e:
        print(f"Error loading data: {e}")
        # If the specified path doesn't work, try looking for CSV files in the current directory
        try:
            # List all files in the current directory
            files_in_dir = os.listdir()
            # Filter for CSV files
            csv_files = [f for f in files_in_dir if f.endswith('.csv')]
            if csv_files:
                print(f"Found CSV files: {csv_files}")
                # Use the first CSV file found
                return pd.read_csv(csv_files[0])
            else:
                print("No CSV files found in the current directory.")
                return None
        except Exception as inner_e:
            print(f"Error listing directory contents: {inner_e}")
            return None

def explore_data(df):
    """
    Explore the dataset and return basic statistics
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The input dataset to explore
        
    Returns:
    --------
    pandas.DataFrame
        The original DataFrame (potentially after cleaning)
        
    Notes:
    ------
    This function prints various exploratory statistics about the dataset:
    - First few rows
    - Dataset dimensions (rows, columns)
    - Data types of each column
    - Summary statistics (mean, std, min, max, quartiles)
    - Count of missing values per column
    """
    # Display the first few rows to understand data structure
    print("Data Sample:")
    print(df.head())
    
    # Show dataset dimensions
    print("\nDataset shape:")
    print(df.shape)
    
    # Display data types of each column
    print("\nData types:")
    print(df.dtypes)
    
    # Calculate summary statistics for numeric columns
    print("\nSummary statistics:")
    print(df.describe())
    
    # Check for missing values in each column
    print("\nMissing values:")
    print(df.isnull().sum())
    
    return df

def feature_correlations(df):
    """
    Calculate and visualize feature correlations
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The input dataset containing features
        
    Returns:
    --------
    pandas.DataFrame
        Correlation matrix of numerical features
        
    Notes:
    ------
    This function:
    - Extracts numerical columns from the dataset
    - Calculates the correlation matrix between all numeric features
    - Creates and saves a heatmap visualization of the correlations
    
    The visualization helps identify potential multicollinearity between features.
    """
    # Select only numeric columns for correlation analysis
    numeric_df = df.select_dtypes(include=[np.number])
    
    # Calculate the correlation matrix between all numeric columns
    corr_matrix = numeric_df.corr()
    
    # Create correlation heatmap visualization
    plt.figure(figsize=(10, 8))
    # Use a heatmap with annotated correlation values
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
    plt.title('Feature Correlation Matrix')
    plt.tight_layout()
    plt.savefig('correlation_heatmap.png')
    plt.close()
    
    return corr_matrix

def prepare_features(df):
    """
    Prepare features and target variable for model training
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The input dataset
        
    Returns:
    --------
    tuple
        X : pandas.DataFrame - Feature matrix
        y : pandas.Series - Target variable
        feature_cols : list - Names of feature columns
        target_col : str - Name of target column
        
    Notes:
    ------
    This function:
    - Identifies numeric columns in the dataset
    - Separates features from the target variable
    - By default, assumes the last numeric column is the target variable
      (This should be adjusted based on your specific dataset)
    """
    # Identify all numeric columns in the dataset
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # Assume the last numeric column is the target variable 
    # Note: This is a common convention but may need to be adjusted for your specific dataset
    target_col = numeric_cols[-1]
    feature_cols = numeric_cols[:-1]
    
    # Print the identified target and feature columns for verification
    print(f"Target variable: {target_col}")
    print(f"Feature variables: {feature_cols}")
    
    # Create the feature matrix X and target vector y
    X = df[feature_cols]  # Features (independent variables)
    y = df[target_col]    # Target (dependent variable)
    
    return X, y, feature_cols, target_col

def check_assumptions(X, y, model, predictions, residuals):
    """
    Check the key assumptions of linear regression
    
    Parameters:
    -----------
    X : pandas.DataFrame
        Feature matrix
    y : pandas.Series
        Target variable
    model : sklearn.linear_model.LinearRegression
        Trained linear regression model
    predictions : numpy.ndarray
        Model predictions
    residuals : numpy.ndarray
        Residuals (actual - predicted values)
        
    Returns:
    --------
    pandas.DataFrame
        Variance Inflation Factors for all features
        
    Notes:
    ------
    This function checks the four main assumptions of linear regression:
    1. Linearity: The relationship between X and y is linear
       - Visualized by plotting predicted values vs residuals
    2. Independence: Residuals are independent of each other
       - Not directly tested here, depends on data collection
    3. Homoscedasticity: Residuals have constant variance
       - Visualized by checking if residuals have consistent spread
    4. Normality: Residuals are normally distributed
       - Visualized with Q-Q plot and histogram
       
    Additionally, this function checks for multicollinearity using VIF.
    """
    # Create a figure with multiple subplots for assumption checking
    plt.figure(figsize=(12, 8))
    
    # 1. Linearity - Plot predicted values vs residuals
    # If linear relationship holds, residuals should be randomly scattered around zero
    plt.subplot(2, 2, 1)
    plt.scatter(predictions, residuals)
    plt.axhline(y=0, color='r', linestyle='-')
    plt.title('Residuals vs Predicted Values')
    plt.xlabel('Predicted Values')
    plt.ylabel('Residuals')
    
    # 2. Normality of residuals - Q-Q plot
    # Points should approximately follow the diagonal line if normally distributed
    plt.subplot(2, 2, 2)
    stats.probplot(residuals, dist="norm", plot=plt)
    plt.title('Q-Q Plot of Residuals')
    
    # 3. Residuals histogram
    # Should approximately follow a bell curve (normal distribution)
    plt.subplot(2, 2, 3)
    plt.hist(residuals, bins=20, edgecolor='black')
    plt.title('Histogram of Residuals')
    plt.xlabel('Residual Value')
    plt.ylabel('Frequency')
    
    # 4. Homoscedasticity - Residuals vs actual values
    # Should show random scatter with consistent spread
    plt.subplot(2, 2, 4)
    plt.scatter(y, residuals)
    plt.axhline(y=0, color='r', linestyle='-')
    plt.title('Residuals vs Actual Values')
    plt.xlabel('Actual Values')
    plt.ylabel('Residuals')
    
    plt.tight_layout()
    plt.savefig('regression_assumptions.png')
    plt.close()
    
    # Check for multicollinearity using Variance Inflation Factor (VIF)
    # VIF > 10 suggests problematic multicollinearity
    vif_data = pd.DataFrame()
    vif_data["Variable"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    print("\nVariance Inflation Factors (VIF):")
    print(vif_data)
    print("Note: VIF > 10 indicates problematic multicollinearity")
    
    return vif_data

def train_linear_regression(X, y):
    """
    Train a linear regression model and evaluate its performance
    
    Parameters:
    -----------
    X : pandas.DataFrame
        Feature matrix
    y : pandas.Series
        Target variable
        
    Returns:
    --------
    tuple
        model : sklearn.linear_model.LinearRegression - Trained model
        X_train, X_test : pandas.DataFrame - Training and testing features
        y_train, y_test : pandas.Series - Training and testing targets
        y_train_pred, y_test_pred : numpy.ndarray - Training and testing predictions
        residuals : numpy.ndarray - Residuals for the full dataset
        all_predictions : numpy.ndarray - Predictions for the full dataset
        
    Notes:
    ------
    This function:
    - Splits data into training (70%) and testing (30%) sets
    - Trains a linear regression model on the training data
    - Evaluates model performance with multiple metrics
    - Creates visualizations of model performance
    - Returns model components for further analysis
    """
    # Split data into training (70%) and testing (30%) sets
    # random_state ensures reproducibility of results
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    
    # Initialize and train the linear regression model
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    # Make predictions on both training and testing sets
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Calculate performance metrics
    # MSE (Mean Squared Error) - Average of squared differences between predictions and actual values
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)
    
    # RMSE (Root Mean Squared Error) - Square root of MSE, in the same units as the target variable
    train_rmse = np.sqrt(train_mse)
    test_rmse = np.sqrt(test_mse)
    
    # R² (Coefficient of Determination) - Proportion of variance explained by the model
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    # Print performance metrics
    print("\nModel Performance:")
    print(f"Training MSE: {train_mse:.4f}")
    print(f"Testing MSE: {test_mse:.4f}")
    print(f"Training RMSE: {train_rmse:.4f}")
    print(f"Testing RMSE: {test_rmse:.4f}")
    print(f"Training R²: {train_r2:.4f}")
    print(f"Testing R²: {test_r2:.4f}")
    
    # Create a DataFrame of model coefficients for each feature
    coefficients = pd.DataFrame({
        'Feature': X.columns,
        'Coefficient': model.coef_
    })
    # Sort coefficients by magnitude for better visualization
    coefficients = coefficients.sort_values(by='Coefficient', ascending=False)
    print("\nModel Coefficients:")
    print(coefficients)
    
    # Plot actual vs predicted values on the test set
    plt.figure(figsize=(10, 6))
    plt.scatter(y_test, y_test_pred, alpha=0.5)
    # Add a diagonal line representing perfect predictions
    plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')
    plt.xlabel('Actual Values')
    plt.ylabel('Predicted Values')
    plt.title('Actual vs Predicted Values')
    plt.savefig('actual_vs_predicted.png')
    plt.close()
    
    # Plot feature importance (coefficients)
    plt.figure(figsize=(12, 8))
    coefficients.plot(kind='bar', x='Feature', y='Coefficient', legend=False)
    plt.title('Feature Coefficients')
    plt.xlabel('Features')
    plt.ylabel('Coefficient Value')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('feature_coefficients.png')
    plt.close()
    
    # Calculate predictions and residuals for the full dataset
    # (needed for assumption checking)
    all_predictions = model.predict(X)
    residuals = y - all_predictions
    
    return model, X_train, X_test, y_train, y_test, y_train_pred, y_test_pred, residuals, all_predictions


    
    # Calculate predictions using the model
    predictions = model.predict(X_sm)
    
    # Calculate confidence intervals for predictions
    # This requires manual calculation
    alpha = 0.05  # 95% confidence level
    n = len(y)    # Sample size
    p = len(model.params)  # Number of parameters (including intercept)
    dof = n - p   # Degrees of freedom
    
    # Get critical value from t-distribution
    t_critical = stats.t.ppf(1 - alpha/2, dof)
    
    # Calculate standard error for predictions
    # This is a simplified version that doesn't account for the full prediction variance
    se_pred = np.sqrt(model.mse_resid * (1 + 1/n))
    
    # Calculate confidence intervals
    ci_lower = predictions - t_critical * se_pred
    ci_upper = predictions + t_critical * se_pred
    
    # Create visualization of predictions with confidence intervals
    plt.figure(figsize=(10, 6))
    
    # Sort values for better visualization
    sorted_indices = np.argsort(y)
    sorted_y = y.iloc[sorted_indices]
    sorted_pred = predictions.iloc[sorted_indices]
    sorted_lower = ci_lower.iloc[sorted_indices]
    sorted_upper = ci_upper.iloc[sorted_indices]
    
    # Plot actual vs predicted values with confidence intervals
    plt.scatter(sorted_y, sorted_pred, alpha=0.5, label='Predictions')
    plt.plot([sorted_y.min(), sorted_y.max()], [sorted_y.min(), sorted_y.max()], 'r--', label='Perfect Fit')
    plt.fill_between(sorted_y, sorted_lower, sorted_upper, color='gray', alpha=0.2, label='95% Confidence Interval')
    
    plt.xlabel(f'Actual {target_col}')
    plt.ylabel(f'Predicted {target_col}')
    plt.title('Predictions with 95% Confidence Interval')
    plt.legend()
    plt.savefig('predictions_with_ci.png')
    plt.close()
    
    return model, results_df

def feature_analysis(X, y, feature_cols, target_col):
    """
    Analyze individual features and their relationships with the target variable
    
    Parameters:
    -----------
    X : pandas.DataFrame
        Feature matrix
    y : pandas.Series
        Target variable
    feature_cols : list
        Names of feature columns
    target_col : str
        Name of target column
        
    Returns:
    --------
    pandas.DataFrame
        DataFrame containing correlations between features and target
        
    Notes:
    ------
    This function:
    - Creates scatter plots of each feature vs. the target variable
    - Adds regression lines to visualize relationships
    - Calculates correlation coefficients and p-values
    - Creates bar charts of correlation strengths
    
    This analysis helps identify which features have the strongest 
    relationships with the target variable.
    """
    # Create a figure for scatter plots of features vs target
    plt.figure(figsize=(15, 10))
    num_features = len(feature_cols)
    
    # Calculate number of rows and columns for subplots
    # (arrange plots in a grid layout)
    n_cols = 3  # Number of columns in the subplot grid
    n_rows = (num_features + n_cols - 1) // n_cols  # Calculate needed rows
    
    # Create scatter plots for each feature vs target
    for i, feature in enumerate(feature_cols):
        plt.subplot(n_rows, n_cols, i + 1)
        
        # Scatter plot of feature vs target
        plt.scatter(X[feature], y, alpha=0.5)
        
        # Add regression line to visualize relationship
        slope, intercept, r_value, p_value, std_err = stats.linregress(X[feature], y)
        line_x = np.array([X[feature].min(), X[feature].max()])
        line_y = intercept + slope * line_x
        plt.plot(line_x, line_y, 'r-')
        
        # Add title with correlation stats
        plt.title(f'{feature} vs {target_col} (r={r_value:.2f}, p={p_value:.3f})')
        plt.xlabel(feature)
        plt.ylabel(target_col)
    
    plt.tight_layout()
    plt.savefig('feature_relationships.png')
    plt.close()
    
    # Calculate correlations, r-values, and p-values for each feature with target
    correlations = []
    for feature in feature_cols:
        # Calculate Pearson correlation coefficient
        corr = X[feature].corr(y)
        
        # Calculate r-value and p-value using scipy.stats
        r_value, p_value = stats.pearsonr(X[feature], y)
        
        correlations.append({
            'Feature': feature,
            'Correlation': corr,
            'R-value': r_value,
            'P-value': p_value
        })
    
    # Create DataFrame of correlations and sort by strength
    corr_df = pd.DataFrame(correlations).sort_values(by='Correlation', ascending=False)
    print("\nFeature Correlations with Target:")
    print(corr_df)
    
    # Create bar chart of correlation strengths
    plt.figure(figsize=(10, 6))
    plt.bar(corr_df['Feature'], corr_df['Correlation'])
    plt.axhline(y=0, color='black', linestyle='-', alpha=0.3)  # Add zero line
    plt.title(f'Feature Correlations with {target_col}')
    plt.xlabel('Features')
    plt.ylabel('Correlation Coefficient')
    plt.xticks(rotation=45, ha='right')  # Rotate labels for readability
    plt.tight_layout()
    plt.savefig('target_correlations.png')
    plt.close()
    
    return corr_df

def detailed_statsmodels_analysis(X, y, feature_cols, target_col):
    """
    Perform detailed statistical analysis using statsmodels
    
    Parameters:
    -----------
    X : pandas.DataFrame
        Feature matrix
    y : pandas.Series
        Target variable
    feature_cols : list
        Names of feature columns
    target_col : str
        Name of target column
        
    Returns:
    --------
    tuple
        sm_model : statsmodels.regression.linear_model.RegressionResultsWrapper - Trained statsmodels model
        results_df : pandas.DataFrame - DataFrame with detailed regression statistics
    """
    # Add a constant term (intercept) to the model
    X_sm = sm.add_constant(X)
    
    # Fit the model using statsmodels
    model = sm.OLS(y, X_sm)
    sm_model = model.fit()
    
    # Print summary of the model
    print(sm_model.summary())
    
    # Create a DataFrame with detailed results
    results_df = pd.DataFrame({
        'Feature': ['constant'] + feature_cols,
        'Coefficient': sm_model.params,
        'Std Error': sm_model.bse,
        't-value': sm_model.tvalues,
        'p-value': sm_model.pvalues,
        'Lower CI': sm_model.conf_int()[0],
        'Upper CI': sm_model.conf_int()[1]
    })
    
    # Calculate VIF (Variance Inflation Factor) for multicollinearity detection
    vif_data = pd.DataFrame()
    vif_data["Feature"] = ['constant'] + feature_cols
    vif_data["VIF"] = [variance_inflation_factor(X_sm.values, i) for i in range(X_sm.shape[1])]
    
    # Merge VIF data with results
    results_df = pd.merge(results_df, vif_data, on="Feature")
    
    # Save detailed results to CSV
    results_df.to_csv('statsmodels_detailed_results.csv', index=False)
    
    # Calculate predictions using the model
    predictions = sm_model.predict(X_sm)
    
    # Calculate confidence intervals for predictions
    # This requires manual calculation
    alpha = 0.05  # 95% confidence level
    n = len(y)    # Sample size
    p = len(sm_model.params)  # Number of parameters (including intercept)
    dof = n - p   # Degrees of freedom
    
    # Get critical value from t-distribution
    t_critical = stats.t.ppf(1 - alpha/2, dof)
    
    # Calculate standard error for predictions
    # This is a simplified version that doesn't account for the full prediction variance
    se_pred = np.sqrt(sm_model.mse_resid * (1 + 1/n))
    
    # Calculate confidence intervals
    ci_lower = predictions - t_critical * se_pred
    ci_upper = predictions + t_critical * se_pred
    
    # Create visualization of predictions with confidence intervals
    plt.figure(figsize=(10, 6))
    
    # Sort values for better visualization
    sorted_indices = np.argsort(y)
    sorted_y = y.iloc[sorted_indices]
    sorted_pred = predictions[sorted_indices]
    sorted_lower = ci_lower[sorted_indices]
    sorted_upper = ci_upper[sorted_indices]
    
    # Plot actual vs predicted values with confidence intervals
    plt.scatter(sorted_y, sorted_pred, alpha=0.5, label='Predictions')
    plt.plot([sorted_y.min(), sorted_y.max()], [sorted_y.min(), sorted_y.max()], 'r--', label='Perfect Fit')
    plt.fill_between(sorted_y, sorted_lower, sorted_upper, color='gray', alpha=0.2, label='95% Confidence Interval')
    
    plt.xlabel(f'Actual {target_col}')
    plt.ylabel(f'Predicted {target_col}')
    plt.title('Predictions with 95% Confidence Interval')
    plt.legend()
    plt.savefig('predictions_with_ci.png')
    plt.close()
    
    return sm_model, results_df

def main():
    """
    Main function to run the complete linear regression analysis
    
    This function coordinates the entire analysis workflow:
    1. Data loading
    2. Exploratory data analysis
    3. Feature preparation
    4. Model training
    5. Assumption checking
    6. Detailed statistical analysis
    
    All outputs (visualizations) are saved as files in the current directory.
    """
    print("Linear Regression Analysis")
    print("=========================")
    
    # Try different possible file paths for the CSV data
    # This handles different environments where the script might be run
    #possible_file_paths = ['data.csv', './data.csv', '../data.csv']
    
    # Attempt to load the data file
    df = None
    # Directly load the file without a loop
    df = load_data(r"D:\Study\UpGrad\AIML\Exercises\Assignment-2_Linear_Regression\Delivery-Starter\Notes\porter_data_1.csv")
    
    # Exit if data loading failed
    if df is None:
        print("Failed to load data. Please check the file path and try again.")
        return
    
    # Step 1: Explore the data
    print("\n=== Exploratory Data Analysis ===")
    df = explore_data(df)
    
    # Step 2: Calculate and visualize feature correlations
    print("\n=== Feature Correlation Analysis ===")
    corr_matrix = feature_correlations(df)
    
    # Step 3: Prepare features and target variable
    print("\n=== Feature and Target Preparation ===")
    X, y, feature_cols, target_col = prepare_features(df)
    
    # Step 4: Analyze relationships between individual features and target
    print("\n=== Feature-Target Relationship Analysis ===")
    feature_corrs = feature_analysis(X, y, feature_cols, target_col)
    
    # Step 5: Train and evaluate the linear regression model
    print("\n=== Linear Regression Model Training and Evaluation ===")
    model, X_train, X_test, y_train, y_test, y_train_pred, y_test_pred, residuals, predictions = train_linear_regression(X, y)
    
    # Step 6: Check linear regression assumptions
    print("\n=== Checking Linear Regression Assumptions ===")
    vif_data = check_assumptions(X, y, model, predictions, residuals)
    
    # Step 7: Perform detailed statistical analysis with statsmodels
    print("\n=== Detailed Statistical Analysis ===")
    sm_model, results_df = detailed_statsmodels_analysis(X, y, feature_cols, target_col)
    
    print("\n=== Analysis complete ===")
    print("Results and visualizations have been saved to the current directory.")

if __name__ == "__main__":
    main()

Linear Regression Analysis
Attempting to load data from: D:\Study\UpGrad\AIML\Exercises\Assignment-2_Linear_Regression\Delivery-Starter\Notes\porter_data_1.csv

=== Exploratory Data Analysis ===
Data Sample:
   market_id           created_at actual_delivery_time  \
0        1.0  2015-02-06 22:24:17  2015-02-06 23:11:17   
1        2.0  2015-02-10 21:49:25  2015-02-10 22:33:25   
2        2.0  2015-02-16 00:11:35  2015-02-16 01:06:35   
3        1.0  2015-02-12 03:36:46  2015-02-12 04:35:46   
4        1.0  2015-01-27 02:12:36  2015-01-27 02:58:36   

   store_primary_category  order_protocol  total_items  subtotal  \
0                       4             1.0            4      3441   
1                      46             2.0            1      1900   
2                      36             3.0            4      4771   
3                      38             1.0            1      1525   
4                      38             1.0            2      3620   

   num_distinct_items  min_item_pr

<Figure size 1200x800 with 0 Axes>